## The Uimin-Lai-Sutherland chain with SU(3) symmetry

The ULS chain is the SU(3)-symmetric spin-1 chain $H = \sum_i P_{i,i+1}$, where $P$ permutes the
two sites. With the eight Gell-Mann generators $\lambda_a$, normalized so that
$\mathrm{tr}(\lambda_a \lambda_b) = 2\delta_{ab}$, the bond reads
$P = \frac{1}{3} + \frac{1}{2}\sum_a \lambda_a \otimes \lambda_a$.

SU(3) is not multiplicity-free ($8 \otimes 8$ contains the adjoint twice), so this chain is the
engine's test of repeated fusion channels: every bond operator is placed through the adjoint
channel of the group.

In [ ]:
from mpsdmrg import Broomstick
from mpsdmrg.tensor import Tensor
from mpsdmrg.symmetry.cat import su3_symmetry
import torch

COMPLEX, CUDA = torch.complex128, torch.device("cuda")
LENGTH, DIM = 6, 3

def gell_mann(device=CUDA):
    '''The eight Hermitian Gell-Mann generators, normalized to tr(T_a T_b) = 2 delta_ab.'''
    raw = [
        [[0, 1, 0], 
         [1, 0, 0], 
         [0, 0, 0]],
        [[0, -1j, 0], 
         [1j, 0, 0], 
         [0, 0, 0]],
        [[1, 0, 0], 
         [0, -1, 0], 
         [0, 0, 0]],
        [[0, 0, 1], 
         [0, 0, 0], 
         [1, 0, 0]],
        [[0, 0, -1j], 
         [0, 0, 0], 
         [1j, 0, 0]],
        [[0, 0, 0], 
         [0, 0, 1], 
         [0, 1, 0]],
        [[0, 0, 0], 
         [0, 0, -1j], 
         [0, 1j, 0]],
        [[1, 0, 0], 
         [0, 1, 0], 
         [0, 0, -2]],
    ]
    return [torch.tensor(m, dtype=COMPLEX, device=device) / (1.0 if i < 7 else 3 ** 0.5)
            for i, m in enumerate(raw)]

def dense_chain(length, dim, bond):
    '''The explicit matrix of the nearest-neighbour bond operator on an open chain.'''
    eye = torch.eye(dim ** (length - 2), dtype=bond.dtype, device=bond.device)
    total = torch.zeros((dim ** length, dim ** length), dtype=bond.dtype, device=bond.device)
    for i in range(length - 1):
        total += torch.kron(torch.kron(torch.eye(dim ** i, dtype=bond.dtype, device=bond.device), bond),
                            torch.eye(dim ** (length - 2 - i), dtype=bond.dtype, device=bond.device))
    return total

generators = gell_mann()
permutation = torch.eye(DIM ** 2, dtype=COMPLEX, device=CUDA) / 3.0 + 0.5 * sum(torch.kron(a, a) for a in generators)
reference = float(torch.linalg.eigvalsh(dense_chain(LENGTH, DIM, permutation)).min())
print('exact ground energy', reference)

## The DMRG run

The eight generators enter the mapping as the adjoint channel of every bond; the
constant $1/3$ is the identity channel. `Broomstick` builds the Hamiltonian, the
SU(3)-invariant start state and the environments on the first `sweep` call.


In [ ]:
mapping = {f'L{i + 1}': Tensor.from_torch(g) for i, g in enumerate(generators)}
mapping['I'] = Tensor.from_torch(torch.eye(DIM, dtype=COMPLEX, device=CUDA))

stick = Broomstick(L=LENGTH, phys_dim=DIM, max_bond_dim=64, svd_tol=1e-14,
                   mapping=mapping, group=su3_symmetry, site_sector=(1, 0))
for i in range(LENGTH - 1):
    stick.add_couplings(('I', 'I'), 1.0 / 3.0, (i, i + 1))
    for name in mapping:
        if name != 'I':
            stick.add_couplings((name, name), 0.5, (i, i + 1))
stick.sweep(num_sweeps=12)

print(f'{"":<14}{"energy":>22}{"per bond":>16}')
print(f'{"exact":<14}{reference:>22.12f}{reference / LENGTH:>16.12f}')
print(f'{"DMRG (SU(3))":<14}{stick.energy:>22.12f}{stick.energy / LENGTH:>16.12f}')
print(f'{"difference":<14}{abs(stick.energy - reference):>22.3e}')
print('bond dims', stick.state.bond_dim)